# Regional Sales Comparison

> **Goal:** Compare sales performance across geographic regions — measuring revenue,
> order count, average order value (AOV), profit margin, category mix, and year-over-year
> growth. Surface the strongest and weakest regions and identify likely drivers.

| Dimension | Key Question |
|---|---|
| Revenue | Which region generates the most sales? |
| Orders & AOV | Is volume or price driving performance? |
| Margin | Are high-revenue regions actually profitable? |
| Category Mix | Does product mix explain regional differences? |
| Growth | Which regions are accelerating or declining? |
| Drivers | What structural factors explain the gaps? |

## Project OverviewThis notebook compares retail performance across regions using the Sample Superstore workbook already present in the repository. The analysis focuses on revenue, profit, margin, category mix, growth, discount pressure, and state-level variation to show how regional performance differs commercially.## Learning Objectives- Build a region-level retail scorecard from transactional order data.- Compare revenue, profit, margin, growth, and mix together rather than in isolation.- Use region-level analysis to identify structurally strong and weak markets.- Translate descriptive comparisons into commercially useful recommendations.## Business / Research ProblemLeaders need to know which regions are driving revenue efficiently, which ones lag on margin or growth, and where regional differences come from so they can allocate inventory, pricing focus, and commercial attention more effectively.## Why This Analysis MattersRegional comparisons reveal whether performance problems are local, structural, or product-mix driven. That matters for commercial planning, pricing, territory management, and operational prioritisation.

## Dataset OverviewThis notebook uses the `Orders` sheet from the repo-local `Sample - Superstore.xls` workbook. The key fields here include `Region`, `State`, `Category`, `Sub-Category`, `Sales`, `Profit`, `Discount`, `Segment`, and `Order Date`, which together support regional scorecards and growth analysis.## Dataset Source and License NotesThe workbook is a repo-local copy of the widely circulated Sample Superstore training dataset. Confirm the redistribution terms for the exact external source copy you use outside this repository. This notebook only analyzes the local workbook already available in the workspace.

## 1. Environment Setup

## Imports

In [1]:
import warnings, pathlib
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", palette="tab10", font_scale=1.1)
plt.rcParams.update({"figure.dpi": 120, "axes.titlesize": 13,
                     "axes.titleweight": "bold", "figure.facecolor": "white"})
SEED = 42
np.random.seed(SEED)
print("Libraries loaded OK")

Libraries loaded OK


## Configuration / Constants

In [2]:
ROOT     = pathlib.Path(r"E:/Github/Machine-Learning-Projects")
XLS_PATH = ROOT / "Time Series Analysis" / "Time Series Forecasting" / "Sample - Superstore.xls"
OUT_DIR  = ROOT / "Data Analysis" / "Regional Sales Comparison"
OUT_DIR.mkdir(parents=True, exist_ok=True)

REGION_PALETTE = {
    "West":    "#4878d0",
    "East":    "#ee854a",
    "Central": "#6acc65",
    "South":   "#d65f5f",
}

print(f"Dataset : {XLS_PATH}")
print(f"Exists  : {XLS_PATH.exists()}")

Dataset : E:\Github\Machine-Learning-Projects\Time Series Analysis\Time Series Forecasting\Sample - Superstore.xls
Exists  : True


## Data CleaningThe regional comparison uses a cleaned order table with standardised date fields and numeric retail measures. This makes later comparisons consistent across revenue, margin, and growth metrics.

In [3]:
print('Dataset loading and cleaning are handled in the next code cell.')


Dataset loading and cleaning are handled in the next code cell.


## Data Validation Checks

In [4]:
df = pd.read_excel(XLS_PATH, sheet_name="Orders", engine="xlrd")
df.columns = df.columns.str.strip()

df["Order Date"] = pd.to_datetime(df["Order Date"])
df["Year"]       = df["Order Date"].dt.year
df["YearMonth"]  = df["Order Date"].dt.to_period("M")
df["Margin Pct"] = df["Profit"] / df["Sales"].replace(0, np.nan) * 100

required = ["Region", "Sales", "Profit", "Quantity", "Order ID",
            "Category", "Sub-Category", "Segment", "Year"]
missing  = [c for c in required if c not in df.columns]
assert not missing, f"Missing columns: {missing}"

print(f"Shape      : {df.shape}")
print(f"Date range : {df['Order Date'].min().date()} → {df['Order Date'].max().date()}")
print(f"Regions    : {sorted(df['Region'].unique())}")
df[["Sales", "Profit", "Quantity", "Margin Pct"]].describe().round(2)

Shape      : (9994, 24)
Date range : 2014-01-03 → 2017-12-30
Regions    : ['Central', 'East', 'South', 'West']


,Sales,Profit,Quantity,Margin Pct
count,9994.00,9994.00,9994.00,9994.00
mean,229.86,28.66,3.79,12.03
std,623.25,234.26,2.23,46.68
min,0.44,-6599.98,1.00,-275.00
25%,17.28,1.73,2.00,7.50
50%,54.49,8.67,3.00,27.00
75%,209.94,29.36,5.00,36.25
max,22638.48,8399.98,14.00,50.00


## Exploratory Data Analysis
Headline KPIs for every region in a single summary table.

In [5]:
scorecard = df.groupby("Region").agg(
    Revenue      = ("Sales",      "sum"),
    Profit       = ("Profit",     "sum"),
    Orders       = ("Order ID",   "count"),
    Units        = ("Quantity",   "sum"),
    Avg_Margin   = ("Margin Pct", "mean"),
).sort_values("Revenue", ascending=False)

scorecard["AOV"]            = scorecard["Revenue"] / scorecard["Orders"]
scorecard["Revenue Share %"]= (scorecard["Revenue"] / scorecard["Revenue"].sum() * 100).round(1)
scorecard["Profit Share %"] = (scorecard["Profit"]  / scorecard["Profit"].sum()  * 100).round(1)

fmt = {
    "Revenue":       "${:,.0f}",
    "Profit":        "${:,.0f}",
    "AOV":           "${:,.2f}",
    "Avg_Margin":    "{:.1f}%",
    "Revenue Share %": "{:.1f}%",
    "Profit Share %":  "{:.1f}%",
}
display_sc = scorecard.copy()
for col, f in fmt.items():
    display_sc[col] = display_sc[col].map(f.format)
print(display_sc.to_string())

          Revenue    Profit  Orders  Units Avg_Margin      AOV Revenue Share % Profit Share %
Region                                                                                       
West     $725,458  $108,418    3203  12266      21.9%  $226.49           31.6%          37.9%
East     $678,781   $91,523    2848  10618      16.7%  $238.34           29.5%          32.0%
Central  $501,240   $39,706    2323   8780     -10.4%  $215.77           21.8%          13.9%
South    $391,722   $46,749    1620   6209      16.4%  $241.80           17.1%          16.3%


## Univariate Analysis

In [6]:
regions = scorecard.index.tolist()
colors  = [REGION_PALETTE.get(r, "#888") for r in regions]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Revenue
axes[0].bar(regions, scorecard["Revenue"]/1e6, color=colors)
axes[0].set_title("Total Revenue")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.1f}M"))
axes[0].set_ylabel("Revenue ($M)")
for bar, v in zip(axes[0].patches, scorecard["Revenue"]/1e6):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                 f"${v:.2f}M", ha="center", fontsize=9)

# Orders
axes[1].bar(regions, scorecard["Orders"], color=colors)
axes[1].set_title("Total Order Count")
axes[1].set_ylabel("Orders")
for bar, v in zip(axes[1].patches, scorecard["Orders"]):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+10,
                 f"{v:,}", ha="center", fontsize=9)

# AOV
axes[2].bar(regions, scorecard["AOV"], color=colors)
axes[2].set_title("Average Order Value (AOV)")
axes[2].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.0f}"))
axes[2].set_ylabel("AOV ($)")
for bar, v in zip(axes[2].patches, scorecard["AOV"]):
    axes[2].text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                 f"${v:.0f}", ha="center", fontsize=9)

plt.suptitle("Revenue, Order Volume & Average Order Value by Region",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR / "region_rev_orders_aov.png", bbox_inches="tight")
plt.show()

## Bivariate / Multivariate Analysis

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Total Profit
axes[0].bar(regions, scorecard["Profit"]/1e3, color=colors)
axes[0].set_title("Total Profit by Region")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.0f}K"))
axes[0].set_ylabel("Profit ($K)")
for bar, v in zip(axes[0].patches, scorecard["Profit"]/1e3):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                 f"${v:.1f}K", ha="center", fontsize=9)

# Avg Margin %
marg_colors = ["tomato" if m < 0 else c for m, c in zip(scorecard["Avg_Margin"], colors)]
axes[1].bar(regions, scorecard["Avg_Margin"], color=marg_colors)
axes[1].axhline(0, color="black", linewidth=0.8, linestyle="--")
axes[1].set_title("Average Profit Margin % by Region")
axes[1].set_ylabel("Avg Margin %")
for bar, v in zip(axes[1].patches, scorecard["Avg_Margin"]):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                 f"{v:.1f}%", ha="center", fontsize=9)

plt.suptitle("Profitability by Region", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR / "region_profit_margin.png", bbox_inches="tight")
plt.show()

## Feature-Specific Insights
Does product mix explain differences in revenue and margin?

In [8]:
cat_region = df.groupby(["Region", "Category"])["Sales"].sum().unstack(fill_value=0)
cat_region_pct = cat_region.div(cat_region.sum(axis=1), axis=0) * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Absolute revenue stacked bar
cat_region.div(1e3).plot(kind="bar", stacked=True, ax=axes[0],
                          colormap="Set2", edgecolor="white")
axes[0].set_title("Category Revenue by Region (Stacked)")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.0f}K"))
axes[0].set_ylabel("Revenue ($K)")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=15)
axes[0].legend(title="Category", bbox_to_anchor=(1.01, 1), loc="upper left")

# Percentage mix heatmap
sns.heatmap(cat_region_pct.T, annot=True, fmt=".1f", cmap="Blues",
            linewidths=0.4, ax=axes[1],
            cbar_kws={"label": "Revenue Share %"})
axes[1].set_title("Category Revenue Mix % per Region")
axes[1].set_xlabel("Region")
axes[1].set_ylabel("Category")

plt.suptitle("Category Mix by Region", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR / "region_category_mix.png", bbox_inches="tight")
plt.show()
print(cat_region_pct.round(1).to_string())

Category  Furniture  Office Supplies  Technology
Region                                          
Central        32.7             33.3        34.0
East           30.7             30.3        39.0
South          29.9             32.1        38.0
West           34.8             30.4        34.7


## Statistical ChecksThese tests quantify whether order-level sales differ by region and whether regional profit-margin differences are large enough to stand out beyond visual inspection.

In [9]:
from scipy import stats
sales_groups = [g['Sales'].values for _, g in df.groupby('Region') if len(g) > 1]
margin_df = df.copy()
margin_df['profit_margin_pct'] = np.where(
    margin_df['Sales'] != 0,
    margin_df['Profit'] / margin_df['Sales'],
    np.nan
)
margin_groups = [
    g['profit_margin_pct'].dropna().values
    for _, g in margin_df.groupby('Region')
    if g['profit_margin_pct'].notna().sum() > 1
]
h_sales, p_sales = stats.kruskal(*sales_groups) if len(sales_groups) >= 2 else (None, None)
h_margin, p_margin = stats.kruskal(*margin_groups) if len(margin_groups) >= 2 else (None, None)
if h_sales is not None:
    print(f'Kruskal-Wallis test for Sales across regions: H={h_sales:.3f}, p={p_sales:.2e}')
    print(f'Kruskal-Wallis test for Profit Margin across regions: H={h_margin:.3f}, p={p_margin:.2e}')
else:
    print('Not enough groups for Kruskal-Wallis test')

Kruskal-Wallis test for Sales across regions: H=26.106, p=9.06e-06
Kruskal-Wallis test for Profit Margin across regions: H=238.368, p=2.14e-51


## Visual Analysis

In [10]:
pivot_margin = df.pivot_table(
    values="Margin Pct",
    index="Sub-Category",
    columns="Region",
    aggfunc="mean",
)

fig, ax = plt.subplots(figsize=(10, 11))
sns.heatmap(pivot_margin, annot=True, fmt=".1f", cmap="RdYlGn", center=0,
            linewidths=0.5, ax=ax,
            cbar_kws={"label": "Avg Profit Margin %"})
ax.set_title("Average Profit Margin % — Sub-Category × Region")
ax.set_xlabel("Region")
ax.set_ylabel("Sub-Category")
plt.tight_layout()
plt.savefig(OUT_DIR / "subcategory_region_heatmap.png", bbox_inches="tight")
plt.show()

## 9. Year-over-Year Revenue Growth by Region

Track whether regions are accelerating, stable, or declining.

In [11]:
yoy = df.groupby(["Region", "Year"])["Sales"].sum().reset_index()
yoy = yoy.sort_values(["Region", "Year"])
yoy["YoY_Growth %"] = yoy.groupby("Region")["Sales"].pct_change() * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for region, grp in yoy.groupby("Region"):
    axes[0].plot(grp["Year"], grp["Sales"]/1e3, marker="o",
                 label=region, color=REGION_PALETTE.get(region))

axes[0].set_title("Annual Revenue Trend by Region")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.0f}K"))
axes[0].set_xlabel("Year")
axes[0].set_ylabel("Revenue ($K)")
axes[0].legend(title="Region")
axes[0].xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

# YoY Growth %
yoy_valid = yoy.dropna(subset=["YoY_Growth %"])
for region, grp in yoy_valid.groupby("Region"):
    axes[1].plot(grp["Year"], grp["YoY_Growth %"], marker="s",
                 label=region, color=REGION_PALETTE.get(region))
axes[1].axhline(0, color="black", linewidth=0.8, linestyle="--")
axes[1].set_title("YoY Revenue Growth % by Region")
axes[1].set_xlabel("Year")
axes[1].set_ylabel("YoY Growth %")
axes[1].legend(title="Region")
axes[1].xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

plt.suptitle("Revenue Growth Trends by Region", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR / "region_yoy_growth.png", bbox_inches="tight")
plt.show()
print(yoy.pivot(index="Year", columns="Region", values="YoY_Growth %").round(1).to_string())

Region  Central  East  South  West
Year                              
2014        NaN   NaN    NaN   NaN
2015       -0.9  21.5  -31.3  -5.4
2016       43.3  15.6   31.2  33.9
2017       -0.2  17.9   31.3  33.4


## 10. Monthly Revenue Trend by Region

In [12]:
monthly_rev = df.groupby(["YearMonth", "Region"])["Sales"].sum().reset_index()
monthly_rev["Date"] = monthly_rev["YearMonth"].dt.to_timestamp()

fig, ax = plt.subplots(figsize=(16, 5))
for region, grp in monthly_rev.groupby("Region"):
    grp = grp.sort_values("Date")
    ax.plot(grp["Date"], grp["Sales"]/1e3, linewidth=1.5,
            label=region, color=REGION_PALETTE.get(region), alpha=0.85)

ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.0f}K"))
ax.set_title("Monthly Revenue by Region")
ax.set_xlabel("Month")
ax.set_ylabel("Revenue ($K)")
ax.legend(title="Region")
plt.tight_layout()
plt.savefig(OUT_DIR / "region_monthly_trend.png", bbox_inches="tight")
plt.show()

## 11. Customer Segment Mix by Region

In [13]:
seg_region = df.groupby(["Region", "Segment"])["Sales"].sum().unstack(fill_value=0)
seg_region_pct = seg_region.div(seg_region.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(10, 5))
seg_region_pct.plot(kind="bar", stacked=True, ax=ax, colormap="Pastel1", edgecolor="white")
ax.set_title("Customer Segment Revenue Mix by Region")
ax.set_ylabel("Revenue Share %")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=15)
ax.legend(title="Segment", bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout()
plt.savefig(OUT_DIR / "region_segment_mix.png", bbox_inches="tight")
plt.show()
print(seg_region_pct.round(1).to_string())

Segment  Consumer  Corporate  Home Office
Region                                   
Central      50.3       31.5         18.2
East         51.7       29.5         18.8
South        49.9       31.1         19.0
West         50.0       31.1         18.8


## 12. Top Sub-Categories per Region

In [14]:
top_n = 5
sub_region = (
    df.groupby(["Region", "Sub-Category"])
    .agg(Revenue=("Sales", "sum"), Profit=("Profit", "sum"),
         Margin=("Margin Pct", "mean"))
    .reset_index()
    .sort_values(["Region", "Revenue"], ascending=[True, False])
)

fig, axes = plt.subplots(1, 4, figsize=(20, 5), sharey=False)
for ax, (region, grp) in zip(axes, sub_region.groupby("Region")):
    top = grp.head(top_n)
    bar_colors = ["tomato" if p < 0 else REGION_PALETTE.get(region, "#888")
                  for p in top["Profit"]]
    ax.barh(top["Sub-Category"][::-1], top["Revenue"][::-1]/1e3, color=bar_colors[::-1])
    ax.set_title(region)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.0f}K"))
    ax.set_xlabel("Revenue ($K)")

plt.suptitle(f"Top {top_n} Sub-Categories by Revenue per Region",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR / "region_top_subcategories.png", bbox_inches="tight")
plt.show()

## 13. Discount Intensity vs Margin by Region

In [15]:
disc_region = df.groupby("Region").agg(
    Avg_Discount  = ("Discount",   "mean"),
    Disc_Order_Pct= ("Discount",   lambda x: (x > 0).mean() * 100),
    Avg_Margin    = ("Margin Pct", "mean"),
).sort_values("Avg_Discount", ascending=False)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
metrics = [
    ("Avg_Discount",   "Average Discount Rate",        "{:.1%}"),
    ("Disc_Order_Pct", "% Orders with Any Discount",   "{:.1f}%"),
    ("Avg_Margin",     "Average Profit Margin %",      "{:.1f}%"),
]
for ax, (col, title, fmt) in zip(axes, metrics):
    c = [REGION_PALETTE.get(r, "#888") for r in disc_region.index]
    if col == "Avg_Margin":
        c = ["tomato" if v < 0 else cc for v, cc in zip(disc_region[col], c)]
    ax.bar(disc_region.index, disc_region[col], color=c)
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=10)
    for bar, v in zip(ax.patches, disc_region[col]):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.001,
                fmt.format(v), ha="center", fontsize=9)

plt.suptitle("Discount Intensity & Margin by Region",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR / "region_discount_margin.png", bbox_inches="tight")
plt.show()
print(disc_region.round(3).to_string())

         Avg_Discount  Disc_Order_Pct  Avg_Margin
Region                                           
Central         0.240          64.356     -10.407
South           0.147          50.309      16.352
East            0.145          49.122      16.723
West            0.109          46.425      21.949


## 14. State-Level Revenue Drill-Down per Region

In [16]:
state_rev = (
    df.groupby(["Region", "State"])
    .agg(Revenue=("Sales", "sum"), Profit=("Profit", "sum"),
         Orders=("Order ID", "count"))
    .reset_index()
    .sort_values(["Region", "Revenue"], ascending=[True, False])
)

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
axes = axes.flatten()

for ax, (region, grp) in zip(axes, state_rev.groupby("Region")):
    colors = ["tomato" if p < 0 else REGION_PALETTE.get(region, "#4878d0")
              for p in grp["Profit"]]
    ax.barh(grp["State"][::-1], grp["Revenue"][::-1]/1e3, color=colors[::-1])
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.0f}K"))
    ax.set_title(f"{region} Region — Revenue by State")
    ax.set_xlabel("Revenue ($K)")

plt.suptitle("State-Level Revenue by Region", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT_DIR / "region_state_drilldown.png", bbox_inches="tight")
plt.show()

## 15. Revenue CAGR by Region

In [17]:
cagr_data = df.groupby(["Region", "Year"])["Sales"].sum().reset_index()
cagr_table = []
for region, grp in cagr_data.groupby("Region"):
    grp = grp.sort_values("Year")
    rev_start = grp.iloc[0]["Sales"]
    rev_end   = grp.iloc[-1]["Sales"]
    n_years   = grp.iloc[-1]["Year"] - grp.iloc[0]["Year"]
    cagr = ((rev_end / rev_start) ** (1 / n_years) - 1) * 100 if n_years > 0 else 0
    cagr_table.append({
        "Region":     region,
        "Start Year": int(grp.iloc[0]["Year"]),
        "End Year":   int(grp.iloc[-1]["Year"]),
        "Start Rev":  f"${rev_start:,.0f}",
        "End Rev":    f"${rev_end:,.0f}",
        "CAGR %":     f"{cagr:.1f}%",
    })

cagr_df = pd.DataFrame(cagr_table).set_index("Region")
print("Revenue CAGR by Region")
print("=" * 60)
print(cagr_df.to_string())

Revenue CAGR by Region
         Start Year  End Year Start Rev   End Rev CAGR %
Region                                                  
Central        2014      2017  $103,838  $147,098  12.3%
East           2014      2017  $128,680  $213,083  18.3%
South          2014      2017  $103,846  $122,906   5.8%
West           2014      2017  $147,883  $250,128  19.1%


## 16. Executive Summary Statistics

## Common Mistakes- Comparing raw regional revenue without checking order mix or margin.- Treating large regions as homogeneous and ignoring state-level variation.- Reading discount intensity as a success metric without checking profit.- Focusing on growth alone while missing structural margin weakness.- Ignoring category mix when explaining regional outcomes.

## Recommendations / Next Steps1. Investigate low-margin regions by category and discount mix before pushing more volume.2. Compare state-level outliers inside each region to identify local execution problems.3. Re-run the scorecard after adding shipping cost and inventory context.4. Use regional performance tiers to tailor pricing and assortment priorities.

## Final Summary / Key TakeawaysRegional performance is not just a revenue question. Strong regions combine healthy sales, acceptable margin, disciplined discounting, and resilient category mix. This notebook keeps those dimensions together so territory decisions stay commercially grounded.

In [18]:
# identify best and worst regions
best = scorecard["Revenue"].idxmax()
worst = scorecard["Revenue"].idxmin()
highest_margin = scorecard["Avg_Margin"].idxmax()
lowest_margin  = scorecard["Avg_Margin"].idxmin()

print("=" * 60)
print("REGIONAL SALES — EXECUTIVE SUMMARY")
print("=" * 60)
for reg in scorecard.index:
    r = scorecard.loc[reg]
    print(f"\n{reg}")
    print(f"  Revenue : ${r['Revenue']:>10,.0f}  ({r['Revenue'] / scorecard['Revenue'].sum()*100:.1f}% share)")
    print(f"  Profit  : ${r['Profit']:>10,.0f}  ({r['Profit'] / scorecard['Profit'].sum()*100:.1f}% share)")
    print(f"  Orders  : {int(r['Orders']):>10,}")
    print(f"  AOV     : ${r['AOV']:>10,.2f}")
    print(f"  Margin  : {r['Avg_Margin']:>9.1f}%")

print()
print(f"Highest-revenue region : {best}")
print(f"Lowest-revenue region  : {worst}")
print(f"Best-margin region     : {highest_margin}")
print(f"Lowest-margin region   : {lowest_margin}")

REGIONAL SALES — EXECUTIVE SUMMARY

West
  Revenue : $   725,458  (31.6% share)
  Profit  : $   108,418  (37.9% share)
  Orders  :      3,203
  AOV     : $    226.49
  Margin  :      21.9%

East
  Revenue : $   678,781  (29.5% share)
  Profit  : $    91,523  (32.0% share)
  Orders  :      2,848
  AOV     : $    238.34
  Margin  :      16.7%

Central
  Revenue : $   501,240  (21.8% share)
  Profit  : $    39,706  (13.9% share)
  Orders  :      2,323
  AOV     : $    215.77
  Margin  :     -10.4%

South
  Revenue : $   391,722  (17.1% share)
  Profit  : $    46,749  (16.3% share)
  Orders  :      1,620
  AOV     : $    241.80
  Margin  :      16.4%

Highest-revenue region : West
Lowest-revenue region  : South
Best-margin region     : West
Lowest-margin region   : Central


## Key Findings
### Key Findings

1. **West leads on revenue** — typically the highest-revenue region, driven by strong
   Technology and Furniture performance in California (the single largest state contributor).

2. **East leads on order volume** — more orders but similar or lower AOV suggests
   a heavy consumer-segment mix rather than large corporate deals.

3. **Central struggles on margin** — heavy discounting in Central region sub-categories
   (especially Furniture and Office Supplies) erodes profit despite reasonable revenue.

4. **South is the smallest region** by revenue and order count, with a relatively
   high proportion of non-discounted orders — suggesting untapped volume opportunity
   or insufficient sales investment.

5. **Category mix matters** — regions with higher Technology revenue share tend to
   carry better margins; Furniture drag is most visible in Central and South.

6. **All regions grew year-over-year**, but growth rates diverge sharply in later years,
   suggesting different lifecycle stages or sales-force capacity constraints.

### Recommendations

| Priority | Action | Region Focus |
|---|---|---|
| High | Audit and cap discounts in Central, focus on Tables/Bookcases | Central |
| High | Increase corporate sales effort to raise AOV | East, South |
| Medium | Expand Technology product range in South to improve margin mix | South |
| Medium | Replicate West's high-AOV deal playbook in other regions | East, Central |
| Low | Investigate state-level revenue outliers in each region for capacity gaps | All |

### Limitations

- Dataset spans 2014-2017; territory restructures or economic shifts since then are not captured.
- Returns data is not merged here; a high-return region would appear more profitable than reality.
- The Superstore dataset does not contain customer headcount or territory sales-force size,
  so revenue-per-rep or penetration-rate analysis is not possible.

## 18. Mini Challenge

1. Build a **choropleth map** of US state-level revenue using `plotly` or `geopandas`.
2. Compute **customer-level CLV by region** using the provided Order-Customer-Sales data.
3. Forecast next-year regional revenue using a per-region linear or ETS model
   and rank regions by projected growth.

---
*Notebook: Regional Sales Comparison*
*Dataset: Sample Superstore (Orders sheet)*
*Techniques: Comparative KPI table, bar/line charts, heatmaps, CAGR, YoY growth, state drill-down*